In [ ]:
import subprocess
from pathlib import Path
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"

if not repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(repo), "checkout", "--detach", commit],
    check=True,
)

print("원본 코드 커밋:",
      subprocess.check_output(
          ["git", "-C", str(repo), "rev-parse", "HEAD"],
          text=True,
      ).strip())
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

원본 코드 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
CUDA 사용 가능: True
GPU: Tesla T4


In [ ]:
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

log_dir = Path("/content/TCN/reproduction_logs")
log_dir.mkdir(parents=True, exist_ok=True)
copy_log = log_dir / (
    f"copy_memory_T1000_seed1111_10epochs_{datetime.now():%Y%m%d_%H%M%S}.log"
)

runner = """
import numpy as np
import runpy
import sys

np.random.seed(1111)
sys.argv = [
    "copymem_test",
    "--blank_len", "1000",
    "--seq_len", "10",
    "--ksize", "8",
    "--levels", "8",
    "--nhid", "10",
    "--dropout", "0.05",
    "--epochs", "10",
    "--seed", "1111",
]
runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")
"""

print("Copy Memory / T=1000 / 시드 1111 / 10에폭")
print("로그 저장 위치:", copy_log)
started = time.perf_counter()

with copy_log.open("w") as log_file:
    process = subprocess.Popen(
        [sys.executable, "-u", "-c", runner],
        cwd="/content/TCN",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
print("로그 파일:", copy_log)

if return_code != 0:
    raise RuntimeError("Copy Memory 실행에 실패했습니다.")

Copy Memory / T=1000 / 시드 1111 / 10에폭
로그 저장 위치: /content/TCN/reproduction_logs/copy_memory_T1000_seed1111_10epochs_20261124_050627.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=10, ksize=8, iters=100, levels=8, blank_len=1000, seq_len=10, log_interval=50, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
| Epoch   1 |    50/  313 batches | lr 0.00050 | ms/batch 47.01 | loss 0.24876210 | accuracy 94.2034
| Epoch   1 |   100/  313 batches | lr 0.00050 | ms/batch  9.98 | loss 0.05640970 | accuracy 98.9066
| Epoch   1 |   150/  313 batches | lr 0.00050 | ms/batch 10.16 | loss 0.03764843 | accuracy 99.0191
| Epoch   1 |   200/  313 batches | lr 0.00050 | ms/batch  9.88 | loss 0.02781480 | accuracy 99.0646
| Epoch   1 |   250

In [ ]:
from google.colab import files

files.download(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_10epochs_20261124_050627.log"
)

In [ ]:
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

log_dir = Path("/content/TCN/reproduction_logs")
log_dir.mkdir(parents=True, exist_ok=True)
copy50_log = log_dir / (
    f"copy_memory_T1000_seed1111_50epochs_{datetime.now():%Y%m%d_%H%M%S}.log"
)

runner = """
import numpy as np
import runpy
import sys
import torch
import torch.nn.functional as F
from pathlib import Path

np.random.seed(1111)
sys.argv = [
    "copymem_test",
    "--blank_len", "1000",
    "--seq_len", "10",
    "--ksize", "8",
    "--levels", "8",
    "--nhid", "10",
    "--dropout", "0.05",
    "--epochs", "50",
    "--seed", "1111",
    "--log-interval", "1000",
]

state = runpy.run_module(
    "TCN.copy_memory.copymem_test",
    run_name="__main__",
)

model = state["model"]
test_x = state["test_x"]
test_y = state["test_y"]
model.eval()

with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    predictions = logits.argmax(dim=-1)

    overall_accuracy = (
        (predictions == test_y).float().mean().item() * 100
    )
    final10_accuracy = (
        (predictions[:, -10:] == test_y[:, -10:])
        .float().mean().item() * 100
    )
    final10_loss = F.cross_entropy(
        logits[:, -10:, :].reshape(-1, 10),
        test_y[:, -10:].reshape(-1),
    ).item()

parameter_count = sum(p.numel() for p in model.parameters())

checkpoint = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_50epochs_checkpoint.pt"
)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "completed_epochs": 50,
        "upstream_commit": "2f8c2b817050206397458dfd1f5a25ce8a32fe65",
    },
    checkpoint,
)

print("FINAL_MODEL_PARAMETERS:", parameter_count)
print(f"FINAL_OVERALL_ACCURACY: {overall_accuracy:.4f}%")
print(f"FINAL_LAST10_ACCURACY: {final10_accuracy:.4f}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {final10_loss:.8f}")
print("CHECKPOINT:", checkpoint)
"""

print("Copy Memory / T=1000 / 시드 1111 / 처음부터 50에폭")
print("로그 저장 위치:", copy50_log)
started = time.perf_counter()

with copy50_log.open("w") as log_file:
    process = subprocess.Popen(
        [sys.executable, "-u", "-c", runner],
        cwd="/content/TCN",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
print("로그 파일:", copy50_log)

if return_code != 0:
    raise RuntimeError("50에폭 실행에 실패했습니다.")

Copy Memory / T=1000 / 시드 1111 / 처음부터 50에폭
로그 저장 위치: /content/TCN/reproduction_logs/copy_memory_T1000_seed1111_50epochs_20261124_051148.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=8, iters=100, levels=8, blank_len=1000, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.02149778  |  Accuracy: 99.1283


Test set: Average loss: 0.02003532  |  Accuracy: 99.1928


Test set: Average loss: 0.01248106  |  Accuracy: 99.4627


Test set: Average loss: 0.00843873  |  Accuracy: 99.7386


Test set: Average loss: 0.00724102  |  Accuracy: 99.8204


Test set: Average loss: 0.01021435  |  Accuracy: 99.3895


Test set: Average loss: 0.00756327  |  Accuracy: 99.6405


Te

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

checkpoint = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_seed1111_50epochs_checkpoint.pt"
)
backup_zip = Path(
    "/content/TCN/reproduction_logs/"
    "copy_memory_T1000_backup_20261124.zip"
)

with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (copy_log, copy50_log, checkpoint):
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, path.name)

print("백업 파일:", backup_zip)
files.download(str(backup_zip))

백업 파일: /content/TCN/reproduction_logs/copy_memory_T1000_backup_20261124.zip


In [ ]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

# 런타임이 재시작됐을 경우를 대비해 원본 커밋 확인
repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 2000
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
copy2000_log = log_dir / f"copy_memory_T{T}_seed1111_50epochs_{stamp}.log"
checkpoint = log_dir / f"copy_memory_T{T}_seed1111_50epochs_checkpoint.pt"

runner = f"""
import numpy as np, runpy, sys, torch
import torch.nn.functional as F

np.random.seed(1111)
sys.argv = ["copymem_test",
    "--blank_len", "{T}", "--seq_len", "10",
    "--ksize", "8", "--levels", "8", "--nhid", "10",
    "--dropout", "0.05", "--epochs", "50", "--seed", "1111",
    "--log-interval", "1000"]
state = runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")

model, test_x, test_y = state["model"], state["test_x"], state["test_y"]
model.eval()
with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    pred = logits.argmax(dim=-1)
    overall = (pred == test_y).float().mean().item() * 100
    last10 = (pred[:, -10:] == test_y[:, -10:]).float().mean().item() * 100
    last10_ce = F.cross_entropy(logits[:, -10:, :].reshape(-1, 10),
                                test_y[:, -10:].reshape(-1)).item()

torch.save({{"model_state_dict": model.state_dict(), "completed_epochs": 50,
            "blank_len": {T}, "upstream_commit": "{commit}"}}, "{checkpoint}")
print("SEQUENCE_TOTAL_LENGTH:", test_y.shape[1])
print("FINAL_MODEL_PARAMETERS:", sum(p.numel() for p in model.parameters()))
print(f"FINAL_OVERALL_ACCURACY: {{overall:.4f}}%")
print(f"FINAL_LAST10_ACCURACY: {{last10:.4f}}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {{last10_ce:.8f}}")
print("CHECKPOINT:", "{checkpoint}")
"""

print(f"Copy Memory / T={T} / 시드 1111 / 처음부터 50에폭")
print("로그:", copy2000_log)
started = time.perf_counter()
with copy2000_log.open("w") as log_file:
    process = subprocess.Popen([sys.executable, "-u", "-c", runner], cwd=str(repo),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line); log_file.flush()
return_code = process.wait()
print("\n종료 코드:", return_code)
print(f"전체 실행 시간: {time.perf_counter() - started:.1f}초")
if return_code != 0:
    raise RuntimeError(f"T={T} 실행에 실패했습니다.")

# 로그 + 체크포인트 백업 ZIP 다운로드
backup_zip = log_dir / f"copy_memory_T{T}_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (copy2000_log, checkpoint):
        archive.write(path, path.name)
print("백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4
Copy Memory / T=2000 / 시드 1111 / 처음부터 50에폭
로그: /content/TCN/reproduction_logs/copy_memory_T2000_seed1111_50epochs_20261124_052742.log
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=8, iters=100, levels=8, blank_len=2000, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.01122813  |  Accuracy: 99.5602


Test set: Average loss: 0.01015619  |  Accuracy: 99.5853


Test set: Average loss: 0.00611462  |  Accuracy: 99.7341


Test set: Average loss: 0.00496229  |  Accuracy: 99.7697


Test set: Average loss: 0.00381203  |  Accuracy: 99.9168


Test set: Average loss: 0.00365342  |  Accuracy: 99.9344


Test 

In [ ]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

script_path = Path("/content/run_copy_memory_recurrent.py")
script_path.write_text(r'''
"""Copy Memory recurrent baselines (GRU/LSTM) using the original TCN data generator."""
import argparse, json, sys, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

parser = argparse.ArgumentParser()
parser.add_argument("--repo", default="/content/TCN")
parser.add_argument("--model", choices=["GRU", "LSTM", "RNN"], required=True)
parser.add_argument("--hidden", type=int, required=True)
parser.add_argument("--blank_len", type=int, default=1000)
parser.add_argument("--seq_len", type=int, default=10)
parser.add_argument("--epochs", type=int, default=50)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--lr", type=float, default=5e-4)
parser.add_argument("--clip", type=float, default=1.0)
parser.add_argument("--seed", type=int, default=1111)
parser.add_argument("--out_prefix", required=True)
args = parser.parse_args()
print(args, flush=True)

sys.path.insert(0, args.repo)
from TCN.copy_memory.utils import data_generator

# TCN 실행과 같은 순서: torch 시드 → NumPy 시드로 학습·테스트 데이터 생성 → 모델 생성
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
np.random.seed(args.seed)
T, seq_len, n_classes = args.blank_len, args.seq_len, 10
train_x, train_y = data_generator(T, seq_len, 10000)
test_x, test_y = data_generator(T, seq_len, 1000)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_x, train_y, test_x, test_y = (t.to(device) for t in (train_x, train_y, test_x, test_y))


class RecurrentBaseline(nn.Module):
    def __init__(self, cell, hidden):
        super().__init__()
        self.rnn = getattr(nn, cell)(1, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, n_classes)

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return self.linear(out)


model = RecurrentBaseline(args.model, args.hidden).to(device)
params = sum(p.numel() for p in model.parameters())
print("MODEL_PARAMETERS:", params, flush=True)
optimizer = torch.optim.RMSprop(model.parameters(), lr=args.lr)


def evaluate():
    model.eval()
    with torch.no_grad():
        logits = model(test_x)
        loss = F.cross_entropy(logits.reshape(-1, n_classes), test_y.reshape(-1)).item()
        pred = logits.argmax(-1)
        overall = (pred == test_y).float().mean().item() * 100
        last10 = (pred[:, -seq_len:] == test_y[:, -seq_len:]).float().mean().item() * 100
        last10_ce = F.cross_entropy(logits[:, -seq_len:].reshape(-1, n_classes),
                                    test_y[:, -seq_len:].reshape(-1)).item()
    return loss, overall, last10, last10_ce


history = []
started = time.perf_counter()
for ep in range(1, args.epochs + 1):
    model.train()
    ep_start = time.perf_counter()
    for start in range(0, train_x.size(0), args.batch_size):
        x = train_x[start:start + args.batch_size]
        y = train_y[start:start + args.batch_size]
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x).reshape(-1, n_classes), y.reshape(-1))
        loss.backward()
        if args.clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)  # backward 뒤 적용
        optimizer.step()
    loss, overall, last10, last10_ce = evaluate()
    history.append({"epoch": ep, "test_loss": loss, "overall_accuracy": overall,
                    "last10_accuracy": last10, "last10_cross_entropy": last10_ce,
                    "epoch_seconds": time.perf_counter() - ep_start})
    print(f"\nTest set: Average loss: {loss:.8f}  |  Accuracy: {overall:.4f}", flush=True)
    print(f"| Epoch {ep:3d} | last10 accuracy {last10:.4f} | last10 CE {last10_ce:.8f} "
          f"| {time.perf_counter() - ep_start:.1f}s", flush=True)

final = history[-1]
print("FINAL_MODEL_PARAMETERS:", params)
print(f"FINAL_OVERALL_ACCURACY: {final['overall_accuracy']:.4f}%")
print(f"FINAL_LAST10_ACCURACY: {final['last10_accuracy']:.4f}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {final['last10_cross_entropy']:.8f}")
print(f"TRAIN_SECONDS: {time.perf_counter() - started:.1f}")
json.dump({"args": vars(args), "parameters": params, "history": history},
          open(args.out_prefix + "_results.json", "w"), indent=2)
torch.save({"model_state_dict": model.state_dict(), "args": vars(args),
            "upstream_commit": "2f8c2b817050206397458dfd1f5a25ce8a32fe65"},
           args.out_prefix + "_checkpoint.pt")
''')

log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs = [script_path]

for model_name, hidden in (("GRU", 61), ("LSTM", 53)):
    prefix = str(log_dir / f"copy_memory_T1000_{model_name.lower()}_seed1111_50epochs_{stamp}")
    log_path = Path(prefix + ".log")
    print(f"\n===== Copy Memory / T=1000 / {model_name} (hidden {hidden}) / 시드 1111 / 50에폭 =====")
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            [sys.executable, "-u", str(script_path), "--model", model_name,
             "--hidden", str(hidden), "--out_prefix", prefix],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    return_code = process.wait()
    print(f"\n{model_name} 종료 코드: {return_code} / 전체 실행 시간: {time.perf_counter() - started:.1f}초")
    outputs += [log_path, Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]
    if return_code != 0:
        raise RuntimeError(f"{model_name} 실행에 실패했습니다.")

backup_zip = log_dir / f"copy_memory_T1000_recurrent_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4

===== Copy Memory / T=1000 / GRU (hidden 61) / 시드 1111 / 50에폭 =====
Namespace(repo='/content/TCN', model='GRU', hidden=61, blank_len=1000, seq_len=10, epochs=50, batch_size=32, lr=0.0005, clip=1.0, seed=1111, out_prefix='/content/TCN/reproduction_logs/copy_memory_T1000_gru_seed1111_50epochs_20261124_053815')
MODEL_PARAMETERS: 12332

Test set: Average loss: 0.02438167  |  Accuracy: 99.1302
| Epoch   1 | last10 accuracy 11.2800 | last10 CE 2.21245646 | 1.9s

Test set: Average loss: 0.02134494  |  Accuracy: 99.1438
| Epoch   2 | last10 accuracy 12.6700 | last10 CE 2.10644722 | 1.3s

Test set: Average loss: 0.02069650  |  Accuracy: 99.1400
| Epoch   3 | last10 accuracy 12.2800 | last10 CE 2.08960986 | 1.3s

Test set: Average loss: 0.02050178  |  Accuracy: 99.1400
| Epoch   4 | last10 accuracy 12.2800 | last10 CE 2.08518195 | 1.3s

Test set: Average loss: 0.02044208  |  Accuracy: 99.1393
| Epoch   5 | last10 accuracy 12.2100 | l

In [ ]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)

mine = Path("/content/tcn-reproduction")
if not mine.exists():
    subprocess.run(["git", "clone", "https://github.com/chamelewan/tcn-reproduction.git", str(mine)], check=True)
else:
    subprocess.run(["git", "-C", str(mine), "pull", "--ff-only"], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("내 저장소 커밋:", subprocess.check_output(["git", "-C", str(mine), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 500
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs = []


def run_and_log(title, cmd, log_path, cwd=None):
    print(f"\n===== {title} =====")
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    code = process.wait()
    print(f"\n종료 코드: {code} / 전체 실행 시간: {time.perf_counter() - started:.1f}초")
    if code != 0:
        raise RuntimeError(f"{title} 실행에 실패했습니다.")


# 1) TCN: 논문 부록 Table 2의 T=500 설정 (커널 6, 블록 9, 채널 10)
tcn_log = log_dir / f"copy_memory_T{T}_tcn_k6_l9_seed1111_50epochs_{stamp}.log"
tcn_ckpt = log_dir / f"copy_memory_T{T}_tcn_k6_l9_seed1111_50epochs_checkpoint.pt"
tcn_runner = f"""
import numpy as np, runpy, sys, torch
import torch.nn.functional as F
np.random.seed(1111)
sys.argv = ["copymem_test", "--blank_len", "{T}", "--seq_len", "10",
    "--ksize", "6", "--levels", "9", "--nhid", "10", "--dropout", "0.05",
    "--epochs", "50", "--seed", "1111", "--log-interval", "1000"]
state = runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")
model, test_x, test_y = state["model"], state["test_x"], state["test_y"]
model.eval()
with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    pred = logits.argmax(dim=-1)
    overall = (pred == test_y).float().mean().item() * 100
    last10 = (pred[:, -10:] == test_y[:, -10:]).float().mean().item() * 100
    last10_ce = F.cross_entropy(logits[:, -10:, :].reshape(-1, 10), test_y[:, -10:].reshape(-1)).item()
torch.save({{"model_state_dict": model.state_dict(), "completed_epochs": 50, "blank_len": {T},
            "ksize": 6, "levels": 9, "upstream_commit": "{commit}"}}, "{tcn_ckpt}")
print("SEQUENCE_TOTAL_LENGTH:", test_y.shape[1])
print("FINAL_MODEL_PARAMETERS:", sum(p.numel() for p in model.parameters()))
print(f"FINAL_OVERALL_ACCURACY: {{overall:.4f}}%")
print(f"FINAL_LAST10_ACCURACY: {{last10:.4f}}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {{last10_ce:.8f}}")
"""
run_and_log(f"Copy Memory / T={T} / TCN (k6, l9, h10) / 시드 1111 / 50에폭",
            [sys.executable, "-u", "-c", tcn_runner], tcn_log, cwd=str(repo))
outputs += [tcn_log, tcn_ckpt]

# 2) GRU·LSTM: T=1000과 같은 직접 구성 기준 모델, 길이만 500
script = mine / "scripts" / "run_copy_memory_recurrent.py"
for model_name, hidden in (("GRU", 61), ("LSTM", 53)):
    prefix = str(log_dir / f"copy_memory_T{T}_{model_name.lower()}_seed1111_50epochs_{stamp}")
    run_and_log(f"Copy Memory / T={T} / {model_name} (hidden {hidden}) / 시드 1111 / 50에폭",
                [sys.executable, "-u", str(script), "--repo", str(repo), "--model", model_name,
                 "--hidden", str(hidden), "--blank_len", str(T), "--out_prefix", prefix],
                Path(prefix + ".log"))
    outputs += [Path(prefix + ".log"), Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]

backup_zip = log_dir / f"copy_memory_T{T}_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
내 저장소 커밋: a141aeaf9eeb54814ae9d4650d84ae763b7e7f22
GPU: Tesla T4

===== Copy Memory / T=500 / TCN (k6, l9, h10) / 시드 1111 / 50에폭 =====
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=6, iters=100, levels=9, blank_len=500, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=1111)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.04187851  |  Accuracy: 98.3550


Test set: Average loss: 0.02809098  |  Accuracy: 98.7521


Test set: Average loss: 0.01822623  |  Accuracy: 99.2677


Test set: Average loss: 0.02002030  |  Accuracy: 98.9292


Test set: Average loss: 0.01311785  |  Accuracy: 99.6437


Test set: Average loss: 0.01154705  |  Accuracy: 99.6983


Test set: Average l

In [ ]:
import subprocess, sys, time, zipfile
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)

mine = Path("/content/tcn-reproduction")
if not mine.exists():
    subprocess.run(["git", "clone", "https://github.com/chamelewan/tcn-reproduction.git", str(mine)], check=True)
else:
    subprocess.run(["git", "-C", str(mine), "pull", "--ff-only"], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("내 저장소 커밋:", subprocess.check_output(["git", "-C", str(mine), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
script = mine / "scripts" / "run_copy_memory_recurrent.py"
outputs = []

for T in (500, 1000):
    prefix = str(log_dir / f"copy_memory_T{T}_rnn_seed1111_50epochs_{stamp}")
    log_path = Path(prefix + ".log")
    print(f"\n===== Copy Memory / T={T} / RNN (hidden 105) / 시드 1111 / 50에폭 =====")
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            [sys.executable, "-u", str(script), "--repo", str(repo), "--model", "RNN",
             "--hidden", "105", "--blank_len", str(T), "--out_prefix", prefix],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    code = process.wait()
    print(f"\nT={T} RNN 종료 코드: {code} / 전체 실행 시간: {time.perf_counter() - started:.1f}초")
    outputs += [log_path, Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]
    if code != 0:
        raise RuntimeError(f"T={T} RNN 실행에 실패했습니다.")

backup_zip = log_dir / f"copy_memory_rnn_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
내 저장소 커밋: cdeeb7f96d342213ffb701e178052dce5c7abeaf
GPU: Tesla T4

===== Copy Memory / T=500 / RNN (hidden 105) / 시드 1111 / 50에폭 =====
Namespace(repo='/content/TCN', model='RNN', hidden=105, blank_len=500, seq_len=10, epochs=50, batch_size=32, lr=0.0005, clip=1.0, seed=1111, out_prefix='/content/TCN/reproduction_logs/copy_memory_T500_rnn_seed1111_50epochs_20261124_055334')
MODEL_PARAMETERS: 12400

Test set: Average loss: 0.04290126  |  Accuracy: 98.2825
| Epoch   1 | last10 accuracy 10.6900 | last10 CE 2.12256408 | 1.1s

Test set: Average loss: 0.04057069  |  Accuracy: 98.3079
| Epoch   2 | last10 accuracy 12.0100 | last10 CE 2.09229803 | 0.7s

Test set: Average loss: 0.04018996  |  Accuracy: 98.3117
| Epoch   3 | last10 accuracy 12.2100 | last10 CE 2.08630347 | 0.7s

Test set: Average loss: 0.04013314  |  Accuracy: 98.3117
| Epoch   4 | last10 accuracy 12.2100 | last10 CE 2.08530617 | 0.9s

Test set: Average loss: 0.04012142  |  Accuracy:

In [ ]:
import subprocess, sys, time, zipfile, re
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
mine = Path("/content/tcn-reproduction")
if not mine.exists():
    subprocess.run(["git", "clone", "https://github.com/chamelewan/tcn-reproduction.git", str(mine)], check=True)
else:
    subprocess.run(["git", "-C", str(mine), "pull", "--ff-only"], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("내 저장소 커밋:", subprocess.check_output(["git", "-C", str(mine), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 1000
SEEDS = (2222, 3333)
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
script = mine / "scripts" / "run_copy_memory_recurrent.py"
outputs, summary = [], []


def run_and_log(title, cmd, log_path, cwd=None):
    print(f"\n===== {title} =====", flush=True)
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    code = process.wait()
    elapsed = time.perf_counter() - started
    print(f"\n종료 코드: {code} / 전체 실행 시간: {elapsed:.1f}초", flush=True)
    if code != 0:
        raise RuntimeError(f"{title} 실행에 실패했습니다.")
    return log_path.read_text(), elapsed


for seed in SEEDS:
    # 1) TCN: 원본 코드, T=1000 설정 (커널 8, 블록 8, 채널 10)
    tcn_log = log_dir / f"copy_memory_T{T}_tcn_seed{seed}_50epochs_{stamp}.log"
    tcn_ckpt = log_dir / f"copy_memory_T{T}_tcn_seed{seed}_50epochs_checkpoint.pt"
    runner = f"""
import numpy as np, runpy, sys, torch
import torch.nn.functional as F
np.random.seed({seed})
sys.argv = ["copymem_test", "--blank_len", "{T}", "--seq_len", "10",
    "--ksize", "8", "--levels", "8", "--nhid", "10", "--dropout", "0.05",
    "--epochs", "50", "--seed", "{seed}", "--log-interval", "1000"]
state = runpy.run_module("TCN.copy_memory.copymem_test", run_name="__main__")
model, test_x, test_y = state["model"], state["test_x"], state["test_y"]
model.eval()
with torch.no_grad():
    logits = model(test_x.unsqueeze(1).contiguous())
    pred = logits.argmax(dim=-1)
    overall = (pred == test_y).float().mean().item() * 100
    last10 = (pred[:, -10:] == test_y[:, -10:]).float().mean().item() * 100
    last10_ce = F.cross_entropy(logits[:, -10:, :].reshape(-1, 10), test_y[:, -10:].reshape(-1)).item()
torch.save({{"model_state_dict": model.state_dict(), "completed_epochs": 50, "blank_len": {T},
            "seed": {seed}, "upstream_commit": "{commit}"}}, "{tcn_ckpt}")
print("FINAL_MODEL_PARAMETERS:", sum(p.numel() for p in model.parameters()))
print(f"FINAL_OVERALL_ACCURACY: {{overall:.4f}}%")
print(f"FINAL_LAST10_ACCURACY: {{last10:.4f}}%")
print(f"FINAL_LAST10_CROSS_ENTROPY: {{last10_ce:.8f}}")
"""
    text, sec = run_and_log(f"Copy Memory / T={T} / TCN / 시드 {seed} / 50에폭",
                            [sys.executable, "-u", "-c", runner], tcn_log, cwd=str(repo))
    outputs += [tcn_log, tcn_ckpt]
    losses = re.findall(r"Average loss: ([0-9.]+)", text)
    summary.append((seed, "TCN", losses[-1], re.search(r"FINAL_LAST10_ACCURACY: ([0-9.]+)", text).group(1), sec))

    # 2) GRU·LSTM·RNN: 직접 구성 기준 모델, 시드만 변경
    for model_name, hidden in (("GRU", 61), ("LSTM", 53), ("RNN", 105)):
        prefix = str(log_dir / f"copy_memory_T{T}_{model_name.lower()}_seed{seed}_50epochs_{stamp}")
        text, sec = run_and_log(
            f"Copy Memory / T={T} / {model_name} (hidden {hidden}) / 시드 {seed} / 50에폭",
            [sys.executable, "-u", str(script), "--repo", str(repo), "--model", model_name,
             "--hidden", str(hidden), "--blank_len", str(T), "--seed", str(seed), "--out_prefix", prefix],
            Path(prefix + ".log"))
        outputs += [Path(prefix + ".log"), Path(prefix + "_results.json"), Path(prefix + "_checkpoint.pt")]
        losses = re.findall(r"Average loss: ([0-9.]+)", text)
        summary.append((seed, model_name, losses[-1], re.search(r"FINAL_LAST10_ACCURACY: ([0-9.]+)", text).group(1), sec))

print("\n===== 요약 (50에폭 최종값) =====")
print(f"{'시드':>6} {'모델':>5} {'테스트 손실':>12} {'마지막10칸':>10} {'시간(초)':>8}")
for seed, m, loss, l10, sec in summary:
    print(f"{seed:>6} {m:>5} {loss:>12} {l10:>9}% {sec:>8.1f}")

backup_zip = log_dir / f"copy_memory_T{T}_multiseed_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
내 저장소 커밋: cf4e9f25562881d514c016b44871b6c82ee26ad5
GPU: Tesla T4

===== Copy Memory / T=1000 / TCN / 시드 2222 / 50에폭 =====
Namespace(batch_size=32, cuda=True, dropout=0.05, clip=1.0, epochs=50, ksize=8, iters=100, levels=8, blank_len=1000, seq_len=10, log_interval=1000, lr=0.0005, optim='RMSprop', nhid=10, seed=2222)
Preparing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.02392315  |  Accuracy: 99.1389


Test set: Average loss: 0.01806691  |  Accuracy: 99.2929


Test set: Average loss: 0.00954788  |  Accuracy: 99.6697


Test set: Average loss: 0.00953337  |  Accuracy: 99.5379


Test set: Average loss: 0.01152377  |  Accuracy: 99.3467


Test set: Average loss: 0.00672778  |  Accuracy: 99.8203


Test set: Average loss: 0.00629

In [ ]:
import subprocess, sys, time, zipfile, re
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T = 600
LRS = ("0.002", "0.001")
SEEDS = (1111, 2222)
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs, summary = [], []

for lr in LRS:
    for seed in SEEDS:
        log_path = log_dir / f"adding_T{T}_tcn_lr{lr}_seed{seed}_10epochs_{stamp}.log"
        # 원본 add_test.py: --cuda를 붙이면 CPU가 되므로 넣지 않음. 기존 T=600 실행과 같은 모델 설정
        runner = f"""
import numpy as np, runpy, sys
np.random.seed({seed})
sys.argv = ["add_test", "--seq_len", "{T}", "--ksize", "8", "--levels", "8", "--nhid", "24",
            "--epochs", "10", "--lr", "{lr}", "--seed", "{seed}"]
runpy.run_module("TCN.adding_problem.add_test", run_name="__main__")
"""
        print(f"\n===== Adding / T={T} / TCN / lr {lr} / 시드 {seed} / 10에폭 =====", flush=True)
        started = time.perf_counter()
        with log_path.open("w") as log_file:
            process = subprocess.Popen([sys.executable, "-u", "-c", runner], cwd=str(repo),
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                if "Train Epoch" not in line:          # 학습 중간 로그는 파일에만 저장
                    print(line, end="", flush=True)
                log_file.write(line); log_file.flush()
        code = process.wait()
        sec = time.perf_counter() - started
        print(f"종료 코드: {code} / 전체 실행 시간: {sec:.1f}초", flush=True)
        if code != 0:
            raise RuntimeError(f"lr {lr} 시드 {seed} 실행에 실패했습니다.")
        mse = [float(v) for v in re.findall(r"Test set: Average loss: ([0-9.]+)", log_path.read_text())]
        first = next((i + 1 for i, v in enumerate(mse) if v < 0.01), None)
        last5 = mse[-5:]
        summary.append((lr, seed, mse[-1], min(mse), mse.index(min(mse)) + 1, first, max(last5) / min(last5), sec))
        outputs.append(log_path)

print("\n===== 요약 (TCN, T=600, 10에폭) =====")
print(f"{'lr':>6} {'시드':>5} {'최종 MSE':>10} {'최저 MSE':>10} {'최저 에폭':>6} {'<0.01 첫 에폭':>12} {'마지막5 흔들림':>12} {'시간(초)':>8}")
for lr, seed, fin, mn, mn_ep, first, ratio, sec in summary:
    print(f"{lr:>6} {seed:>5} {fin:>10.6f} {mn:>10.6f} {mn_ep:>6} {str(first):>12} {ratio:>12.2f} {sec:>8.1f}")
print("\n비교 기준 (기존 lr 0.004): 시드 1111 최종 0.000659 / 시드 2222 최종 0.000102, GRU 약 0.000027")

backup_zip = log_dir / f"adding_T{T}_tcn_lr_sweep_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4

===== Adding / T=600 / TCN / lr 0.002 / 시드 1111 / 10에폭 =====
Namespace(batch_size=32, cuda=True, dropout=0.0, clip=-1, epochs=10, ksize=8, levels=8, seq_len=600, log_interval=100, lr=0.002, optim='Adam', nhid=24, seed=1111)
Producing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.179989


Test set: Average loss: 0.002185


Test set: Average loss: 0.001637


Test set: Average loss: 0.000434


Test set: Average loss: 0.000361


Test set: Average loss: 0.000940


Test set: Average loss: 0.000419


Test set: Average loss: 0.000824


Test set: Average loss: 0.000855


Test set: Average loss: 0.000706

종료 코드: 0 / 전체 실행 시간: 178.8초

===== Adding / T=600 / TCN / lr 0.002 / 시드 2222 / 10에폭 =====
Namespace(batch_size=32, 

In [ ]:
import subprocess, sys, time, zipfile, re
from pathlib import Path
from datetime import datetime
import torch

repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

T, EPOCHS = 600, 30
LRS = ("0.002", "0.001")
SEEDS = (1111, 2222)
log_dir = repo / "reproduction_logs"
log_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs, summary = [], []

for lr in LRS:
    for seed in SEEDS:
        log_path = log_dir / f"adding_T{T}_tcn_lr{lr}_seed{seed}_{EPOCHS}epochs_{stamp}.log"
        runner = f"""
import numpy as np, runpy, sys
np.random.seed({seed})
sys.argv = ["add_test", "--seq_len", "{T}", "--ksize", "8", "--levels", "8", "--nhid", "24",
            "--epochs", "{EPOCHS}", "--lr", "{lr}", "--seed", "{seed}"]
runpy.run_module("TCN.adding_problem.add_test", run_name="__main__")
"""
        print(f"\n===== Adding / T={T} / TCN / lr {lr} / 시드 {seed} / {EPOCHS}에폭 =====", flush=True)
        started = time.perf_counter()
        with log_path.open("w") as log_file:
            process = subprocess.Popen([sys.executable, "-u", "-c", runner], cwd=str(repo),
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                if "Train Epoch" not in line:
                    print(line, end="", flush=True)
                log_file.write(line); log_file.flush()
        code = process.wait()
        sec = time.perf_counter() - started
        print(f"종료 코드: {code} / 전체 실행 시간: {sec:.1f}초", flush=True)
        if code != 0:
            raise RuntimeError(f"lr {lr} 시드 {seed} 실행에 실패했습니다.")
        mse = [float(v) for v in re.findall(r"Test set: Average loss: ([0-9.]+)", log_path.read_text())]
        summary.append((lr, seed, mse[9], mse[19], mse[-1], min(mse), mse.index(min(mse)) + 1, sec))
        outputs.append(log_path)

print(f"\n===== 요약 (TCN, T={T}, {EPOCHS}에폭) =====")
print(f"{'lr':>6} {'시드':>5} {'10에폭':>10} {'20에폭':>10} {'30에폭':>10} {'최저':>10} {'최저 에폭':>6} {'시간(초)':>8}")
for lr, seed, e10, e20, e30, mn, mn_ep, sec in summary:
    print(f"{lr:>6} {seed:>5} {e10:>10.6f} {e20:>10.6f} {e30:>10.6f} {mn:>10.6f} {mn_ep:>6} {sec:>8.1f}")
print("\n비교 기준: GRU (lr 0.004, 10에폭) 시드 1111 0.000027 / 시드 2222 0.000026")

backup_zip = log_dir / f"adding_T{T}_tcn_{EPOCHS}epochs_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
GPU: Tesla T4

===== Adding / T=600 / TCN / lr 0.002 / 시드 1111 / 30에폭 =====
Namespace(batch_size=32, cuda=True, dropout=0.0, clip=-1, epochs=30, ksize=8, levels=8, seq_len=600, log_interval=100, lr=0.002, optim='Adam', nhid=24, seed=1111)
Producing data...
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

Test set: Average loss: 0.180044


Test set: Average loss: 0.001033


Test set: Average loss: 0.001177


Test set: Average loss: 0.000566


Test set: Average loss: 0.000670


Test set: Average loss: 0.000949


Test set: Average loss: 0.000665


Test set: Average loss: 0.000373


Test set: Average loss: 0.000368


Test set: Average loss: 0.000286


Test set: Average loss: 0.000143


Test set: Average loss: 0.000261


Test set: Average loss: 0.000114


Test set:

In [4]:
import subprocess, sys, time, zipfile, re, json, statistics
from pathlib import Path
from datetime import datetime
import torch

assert torch.cuda.is_available(), "GPU가 연결되지 않았습니다. 런타임 유형을 T4로 바꾼 뒤 다시 실행하세요."
repo = Path("/content/TCN")
commit = "2f8c2b817050206397458dfd1f5a25ce8a32fe65"
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/locuslab/TCN.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
mine = Path("/content/tcn-reproduction")
if not mine.exists():
    subprocess.run(["git", "clone", "https://github.com/chamelewan/tcn-reproduction.git", str(mine)], check=True)
else:
    subprocess.run(["git", "-C", str(mine), "pull", "--ff-only"], check=True)
print("원본 커밋:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())
print("내 저장소 커밋:", subprocess.check_output(["git", "-C", str(mine), "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0))

EPOCHS, SEEDS = 30, (1111, 2222)
out_dir = Path("/content/gru30_logs"); out_dir.mkdir(parents=True, exist_ok=True)
stamp = f"{datetime.now():%Y%m%d_%H%M%S}"
outputs, summary = [], []
TCN_MED = {1111: 5.4e-5, 2222: 8.5e-5}   # TCN lr 0.001, 30에폭, 마지막 5에폭 중앙값 (기존 로그에서 계산)
GRU10_PREV = {1111: 2.71e-5, 2222: 2.57e-5}  # 기존 GRU 10에폭 결과 (재현 확인용)

for seed in SEEDS:
    log_path = out_dir / f"adding_T600_gru_seed{seed}_{EPOCHS}epochs_{stamp}.log"
    print(f"\n===== Adding / T=600 / GRU / lr 0.004 / 시드 {seed} / {EPOCHS}에폭 =====", flush=True)
    started = time.perf_counter()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            [sys.executable, "-u", str(mine / "scripts" / "run_t600_recurrent.py"), "--repo", str(repo),
             "--model", "GRU", "--seed", str(seed), "--epochs", str(EPOCHS), "--outdir", str(out_dir)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line); log_file.flush()
    code = process.wait(); sec = time.perf_counter() - started
    print(f"종료 코드: {code} / 전체 실행 시간: {sec:.1f}초", flush=True)
    if code != 0:
        raise RuntimeError(f"GRU 시드 {seed} 실행에 실패했습니다.")
    mse = [float(v) for v in re.findall(r"test_mse=([0-9.e-]+)", log_path.read_text())]
    med = statistics.median(mse[-5:])
    summary.append((seed, mse[9], mse[-1], med, min(mse), mse.index(min(mse)) + 1, TCN_MED[seed] / med, sec))
    outputs += [log_path, out_dir / f"adding_gru_T600_seed{seed}_{EPOCHS}epochs_results.json"]

print(f"\n===== 요약 (GRU, T=600, {EPOCHS}에폭) =====")
print(f"{'시드':>5} {'10에폭':>10} {'(기존10)':>10} {'30에폭':>10} {'마지막5 중앙값':>14} {'최저':>10} {'최저 에폭':>6} {'R=TCN/GRU':>10} {'시간(초)':>8}")
for seed, e10, e30, med, mn, mn_ep, R, sec in summary:
    print(f"{seed:>5} {e10:>10.2e} {GRU10_PREV[seed]:>10.2e} {e30:>10.2e} {med:>14.2e} {mn:>10.2e} {mn_ep:>6} {R:>10.2f} {sec:>8.1f}")
print("판정 기준: R≥2 두 시드 → GRU 우위 / 0.5<R<2 두 시드 → 대등 / R≤0.5 두 시드 → TCN 우위 / 그 외 보류")

backup_zip = out_dir / f"adding_T600_gru_{EPOCHS}epochs_backup_{stamp}.zip"
with zipfile.ZipFile(backup_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in outputs:
        archive.write(path, path.name)
print("\n백업 파일:", backup_zip)
from google.colab import files
files.download(str(backup_zip))

원본 커밋: 2f8c2b817050206397458dfd1f5a25ce8a32fe65
내 저장소 커밋: e102d38caf844f9b10b73fff5dd4dda4915c6dd7
GPU: Tesla T4

===== Adding / T=600 / GRU / lr 0.004 / 시드 1111 / 30에폭 =====
model=GRU seed=1111 params=69451 device=cuda
constant_one_test_mse=0.181991279
epoch=1 test_mse=0.000365376
epoch=2 test_mse=0.000203267
epoch=3 test_mse=0.000114776
epoch=4 test_mse=0.000017126
epoch=5 test_mse=0.000018908
epoch=6 test_mse=0.000022794
epoch=7 test_mse=0.000016220
epoch=8 test_mse=0.000015628
epoch=9 test_mse=0.000023678
epoch=10 test_mse=0.000027066
epoch=11 test_mse=0.000016606
epoch=12 test_mse=0.000009863
epoch=13 test_mse=0.000019533
epoch=14 test_mse=0.000006429
epoch=15 test_mse=0.000005743
epoch=16 test_mse=0.000002935
epoch=17 test_mse=0.000008881
epoch=18 test_mse=0.000010528
epoch=19 test_mse=0.000011887
epoch=20 test_mse=0.000003245
epoch=21 test_mse=0.000004206
epoch=22 test_mse=0.000009783
epoch=23 test_mse=0.000003449
epoch=24 test_mse=0.000003303
epoch=25 test_mse=0.000086849
epoch

In [3]:
import subprocess, torch
print("PyTorch:", torch.__version__, "/ CUDA 빌드:", torch.version.cuda)
print("GPU 사용 가능:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi 없음 (GPU 미연결)")

PyTorch: 2.14.1+cu130 / CUDA 빌드: 13.0
GPU 사용 가능: True
GPU 0: Tesla T4 (UUID: GPU-a40a223a-dc6b-7747-4df4-110076990772)

